In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- rft_concat_write ---
RFT_CONCAT_BASE_PD = pd.DataFrame({"Realization":[1,2],"Well":["W1","W1"],"Ensemble":["E1","E1"],"Iteration":[1,1],"value":[5.0,6.0]})
RFT_CONCAT_BASE_PL = pl.from_pandas(RFT_CONCAT_BASE_PD)
FIX_RFT_CONCAT_WRITE_DROP_CONST_COLS = False
FIX_RFT_CONCAT_WRITE_OUTPUT_FILE = "test_file.csv"

def make_rft_concat_write_data_pd():
    return [RFT_CONCAT_BASE_PD.copy()]

def make_rft_concat_write_data_pl():
    return [RFT_CONCAT_BASE_PL.clone()]

# --- rft_obs_join ---
FIX_RFT_OBS_JOIN_OBS_NODE = pd.DataFrame({"observations": [10.0], "std": [0.5]})
FIX_RFT_OBS_JOIN_PRESSURE_VALS = np.array([1.0, 2.0, 3.0])
FIX_RFT_OBS_JOIN_REALIZATIONS = [0]
FIX_RFT_OBS_JOIN_TVD_ARG = [100., 200., 300.]
FIX_RFT_OBS_JOIN_WELL = "W1"

def make_rft_obs_join_data_pd():
    return []

def make_rft_obs_join_data_pl():
    return []

def make_rft_obs_join_realization_frame_pd():
    return pd.DataFrame({"date":["2020-01-01"],"value":[100.0]})

def make_rft_obs_join_realization_frame_pl():
    return pl.DataFrame({"date":["2020-01-01"],"value":[100.0]})

print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_rft_concat_write(data, drop_const_cols, output_file):
    frame = pd.concat(data)
    frame.set_index(["Realization", "Well", "Ensemble", "Iteration"], inplace=True)
    if drop_const_cols:
        frame = frame.loc[:, (frame != frame.iloc[0]).any()]
    frame.to_csv(output_file)
    return frame

def before_rft_obs_join(data, obs_node, pressure_vals, realizations, tvd_arg, well, realization_frame):
    rft_data = pd.DataFrame(pressure_vals, index=range(len(tvd_arg)))
    ensemble_data = []
    for iens in realizations:
        frame = pd.DataFrame(
            data={"TVD": tvd_arg, "Pressure": rft_data[iens],
                  "ObsValue": obs_node["observations"].values[0],
                  "ObsStd": obs_node["std"].values[0]},
        )
        realization_frame["Realization"] = iens
        realization_frame["Well"] = well
        ensemble_data.append(realization_frame)
    data.append(pd.concat(ensemble_data))
    return ensemble_data

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_rft_concat_write(data, drop_const_cols, output_file):

    frame = pl.concat(data)
    index_cols = ["Realization", "Well", "Ensemble", "Iteration"]
    frame = frame.select(index_cols + [c for c in frame.columns if c not in index_cols])
    if drop_const_cols:
        first_row = frame.row(0)
        data_cols = frame.columns[len(index_cols):]
        keep_cols = index_cols + [
            c
            for c, v in zip(data_cols, first_row[len(index_cols):])
            if frame.get_column(c).ne(v).fill_null(True).any()
        ]
        frame = frame.select(keep_cols)
    frame.write_csv(output_file)
    return frame

def gen_rft_obs_join(data, obs_node, pressure_vals, realizations, tvd_arg, well, realization_frame):

    rft_data = pl.DataFrame(pressure_vals)
    ensemble_data = []
    for iens in realizations:
        realization_frame = pl.DataFrame(
            {
                "TVD": tvd_arg,
                "Pressure": rft_data.get_column(iens),
                "ObsValue": obs_node["observations"].to_list()[0],
                "ObsStd": obs_node["std"].to_list()[0],
            }
        )
        realization_frame = realization_frame.with_columns(
            pl.lit(iens).alias("Realization"),
            pl.lit(well).alias("Well"),
        )
        ensemble_data.append(realization_frame)
    data.append(pl.concat(ensemble_data))
    return ensemble_data

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def _comparison_label(label):
    text = str(label)
    if text.lstrip().startswith(("L2", "L3")):
        return text
    return f"L2 equivalence {text}"

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: rft_obs_join ===

# L1 smoke – generated
try:
    _r = gen_rft_obs_join(make_rft_obs_join_data_pl(), FIX_RFT_OBS_JOIN_OBS_NODE, FIX_RFT_OBS_JOIN_PRESSURE_VALS, FIX_RFT_OBS_JOIN_REALIZATIONS, FIX_RFT_OBS_JOIN_TVD_ARG, FIX_RFT_OBS_JOIN_WELL, make_rft_obs_join_realization_frame_pl())
    print("✅ L1 smoke gen_rft_obs_join: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_rft_obs_join: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_rft_obs_join(make_rft_obs_join_data_pd(), FIX_RFT_OBS_JOIN_OBS_NODE, FIX_RFT_OBS_JOIN_PRESSURE_VALS, FIX_RFT_OBS_JOIN_REALIZATIONS, FIX_RFT_OBS_JOIN_TVD_ARG, FIX_RFT_OBS_JOIN_WELL, make_rft_obs_join_realization_frame_pd())
    print("✅ L1 smoke before_rft_obs_join: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_rft_obs_join: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_rft_obs_join(make_rft_obs_join_data_pd(), FIX_RFT_OBS_JOIN_OBS_NODE, FIX_RFT_OBS_JOIN_PRESSURE_VALS, FIX_RFT_OBS_JOIN_REALIZATIONS, FIX_RFT_OBS_JOIN_TVD_ARG, FIX_RFT_OBS_JOIN_WELL, make_rft_obs_join_realization_frame_pd())
    _rg = gen_rft_obs_join(make_rft_obs_join_data_pl(), FIX_RFT_OBS_JOIN_OBS_NODE, FIX_RFT_OBS_JOIN_PRESSURE_VALS, FIX_RFT_OBS_JOIN_REALIZATIONS, FIX_RFT_OBS_JOIN_TVD_ARG, FIX_RFT_OBS_JOIN_WELL, make_rft_obs_join_realization_frame_pl())
    compare(pd.concat(_rb), pl.concat(_rg), "rft_obs_join")
except Exception as _e:
    print(f"❌ L2 equivalence rft_obs_join: setup error — {type(_e).__name__}: {_e}")

# L3 edge – no realizations should fail consistently at concat setup
try:
    try:
        before_rft_obs_join(make_rft_obs_join_data_pd(), FIX_RFT_OBS_JOIN_OBS_NODE, FIX_RFT_OBS_JOIN_PRESSURE_VALS, [], FIX_RFT_OBS_JOIN_TVD_ARG, FIX_RFT_OBS_JOIN_WELL, make_rft_obs_join_realization_frame_pd())
        _before_exc = None
    except Exception as _e:
        _before_exc = type(_e).__name__
    try:
        gen_rft_obs_join(make_rft_obs_join_data_pl(), FIX_RFT_OBS_JOIN_OBS_NODE, FIX_RFT_OBS_JOIN_PRESSURE_VALS, [], FIX_RFT_OBS_JOIN_TVD_ARG, FIX_RFT_OBS_JOIN_WELL, make_rft_obs_join_realization_frame_pl())
        _gen_exc = None
    except Exception as _e:
        _gen_exc = type(_e).__name__
    if _before_exc and _gen_exc and _gen_exc not in ("SyntaxError", "NameError"):
        print(f"✅ L3 edge rft_obs_join no realizations: both sides reject (before={_before_exc}, gen={_gen_exc})")
    else:
        print(f"❌ L3 edge rft_obs_join no realizations: MISMATCH — before={_before_exc}, gen={_gen_exc}")
except Exception as _e:
    print(f"❌ L3 edge rft_obs_join: {type(_e).__name__}: {_e}")

# AUDIT-72: compare mutation of the caller-provided data list.
try:
    _before_data, _gen_data = [], []
    _rb = before_rft_obs_join(_before_data, FIX_RFT_OBS_JOIN_OBS_NODE, FIX_RFT_OBS_JOIN_PRESSURE_VALS, FIX_RFT_OBS_JOIN_REALIZATIONS, FIX_RFT_OBS_JOIN_TVD_ARG, FIX_RFT_OBS_JOIN_WELL, make_rft_obs_join_realization_frame_pd())
    _rg = gen_rft_obs_join(_gen_data, FIX_RFT_OBS_JOIN_OBS_NODE, FIX_RFT_OBS_JOIN_PRESSURE_VALS, FIX_RFT_OBS_JOIN_REALIZATIONS, FIX_RFT_OBS_JOIN_TVD_ARG, FIX_RFT_OBS_JOIN_WELL, make_rft_obs_join_realization_frame_pl())
    compare(pd.concat(_rb), pl.concat(_rg), "rft_obs_join returned frames", check_row_order=True)
    compare(_before_data[-1], _gen_data[-1], "rft_obs_join appended data", check_row_order=True)
except Exception as _e:
    print(f"❌ L2 equivalence rft_obs_join side effect: {type(_e).__name__}: {_e}")


❌ L1 smoke gen_rft_obs_join: TypeError: argument 'name': 'int' object cannot be cast as 'str'
✅ L1 smoke before_rft_obs_join: OK
❌ L2 equivalence rft_obs_join: setup error — TypeError: argument 'name': 'int' object cannot be cast as 'str'
✅ L3 edge rft_obs_join no realizations: both sides reject (before=ValueError, gen=ValueError)
❌ L2 equivalence rft_obs_join side effect: TypeError: argument 'name': 'int' object cannot be cast as 'str'
